In [2]:
import pandas as pd
import numpy as np

# ── Load raw data ─────────────────────────────────────────────────────────
df1 = pd.read_csv("../exported_data/graph_edges_dataset.csv");        df1['source'] = 'scam'
df2 = pd.read_csv("../exported_data/graph_edges_dataset_2.csv");      df2['source'] = 'scam'
df3 = pd.read_csv("../exported_data/graph_edges_safe_dataset.csv");   df3['source'] = 'safe'
df4 = pd.read_csv("../exported_data/graph_edges_safe_dataset_3.csv"); df4['source'] = 'safe'

edges_df   = pd.concat([df1, df2, df3, df4], axis=0).reset_index(drop=True)
origins_df = pd.read_csv("../exported_data/wallet_genesis_origins.csv")

# ── Rebuild puppet_wallets ────────────────────────────────────────────────
funder_counts     = origins_df['Funder_Wallet'].value_counts()
malicious_funders = funder_counts[funder_counts > 2].index.tolist()
malicious_funders = [f for f in malicious_funders if str(f) not in ('Unknown', 'nan')]
puppet_wallets    = set(origins_df[origins_df['Funder_Wallet'].isin(malicious_funders)]['Wallet'].tolist())

print(f"Edges:           {len(edges_df):,}")
print(f"Wallets crawled: {len(origins_df):,}")
print(f"Puppet wallets:  {len(puppet_wallets):,}")

# ── Synthetic features for CRAWLED wallets ────────────────────────────────
np.random.seed(42)
BASE_EPOCH = 1704067200  # Jan 1 2024 UTC

unique_funders     = origins_df['Funder_Wallet'].unique()
funder_base_time   = {f: BASE_EPOCH + int(np.random.uniform(0, 120*24*3600)) for f in unique_funders}
funder_base_amount = {f: round(np.random.uniform(0.05, 0.5), 4) for f in unique_funders}

def gen_blocktime(funder):
    if str(funder) in ('Unknown', 'nan'):
        return int(np.random.uniform(BASE_EPOCH, BASE_EPOCH + 365*24*3600))
    #  widened: ±7 days instead of ±30 sec — flat models can't cluster on time alone
    return funder_base_time[funder] + int(np.random.uniform(0, 7*24*3600))

def gen_funding_amount(funder):
    if str(funder) in ('Unknown', 'nan'):
        return round(np.random.uniform(0.5, 20.0), 4)
    base = funder_base_amount[funder]
    # widened: ±0.15 SOL instead of ±0.005 — amounts overlap with organic wallets
    return round(base + np.random.uniform(-0.15, 0.15), 6)


origins_df['blockTime']      = origins_df['Funder_Wallet'].apply(gen_blocktime)
origins_df['Funding_Amount'] = origins_df['Funder_Wallet'].apply(gen_funding_amount)

# ── Synthetic features for UNCRAWLED wallets ──────────────────────────────
all_wallets     = set(edges_df['Sender'].tolist() + edges_df['Receiver'].tolist())
crawled_wallets = set(origins_df['Wallet'].tolist())
uncrawled       = all_wallets - crawled_wallets

print(f"\nUncrawled wallets: {len(uncrawled):,} → all treated as organic (no proof otherwise)")

# ALL uncrawled → organic-like wide-spread features (we cannot confirm any as scam)
synthetic_rows = []
for w in uncrawled:
    synthetic_rows.append({
        'Wallet':            w,
        'Genesis_Signature': 'synthetic',
        'Funder_Wallet':     'Unknown',
        'blockTime':         int(np.random.uniform(BASE_EPOCH, BASE_EPOCH + 365*24*3600)),
        'Funding_Amount':    round(np.random.uniform(0.5, 20.0), 4)
    })

# ── Merge and save ────────────────────────────────────────────────────────
origins_full = pd.concat([origins_df, pd.DataFrame(synthetic_rows)], ignore_index=True)
origins_full.to_csv("../exported_data/wallet_genesis_origins_enriched.csv", index=False)

# Reload lookup dicts
wallet_blocktime = origins_full.set_index('Wallet')['blockTime'].to_dict()
wallet_funding   = origins_full.set_index('Wallet')['Funding_Amount'].to_dict()

print(f" Saved enriched CSV — {len(origins_full):,} total wallets")
print(f"   Crawled:    {len(origins_df):,}")
print(f"   Synthetic:  {len(uncrawled):,}")

# ── Sanity check: scam ring tightness ─────────────────────────────────────
scam_sample = origins_full[origins_full['Wallet'].isin(puppet_wallets)].groupby('Funder_Wallet').head(3)
print("\nSample scam ring (same funder = tight times + similar amounts):")
print(scam_sample[['Wallet','Funder_Wallet','blockTime','Funding_Amount']].head(9).to_string(index=False))


Edges:           29,637
Wallets crawled: 3,268
Puppet wallets:  1,730

Uncrawled wallets: 7,832 → all treated as organic (no proof otherwise)
 Saved enriched CSV — 11,100 total wallets
   Crawled:    3,268
   Synthetic:  7,832

Sample scam ring (same funder = tight times + similar amounts):
                                      Wallet                                Funder_Wallet  blockTime  Funding_Amount
5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9 5tzFkiKscXHK5ZXCGbXZxdw7gTjjD1mBwuoFbhUvuAi9 1706862147        0.363945
EJ8QTGejSKHHr42jUYizQA5riAtXh43ZrzgRYMHJqRqY GT2zuHVaZQYZSyQMgJPLzvkmyztfyXg2NJunqFp4p3A4 1706067338        0.075414
65ZHSArs5XxPseKQbB1B4r16vDxMWnCxHMzogDAqiDUc CRo8DBwrmd97DJfAnvCv96tZPL5Mktf2NZy2ZnhDer1A 1706448098        0.007446
F79w9myaPR3jGU2486qT8pSoZjpKnbfkcWpykm6J6iYB AeBwztwXScyNNuQCEdhS54wttRQrw3Nj1UtqddzB4C7b 1709126295        0.290230
7wAC5v4uwByc9RbqJ3hbB3edKosrMmi4Udjv5RtoWEQy F79w9myaPR3jGU2486qT8pSoZjpKnbfkcWpykm6J6iYB 1707402041        0.164258
HJcrk5

In [17]:
df4

,Token,Signature,Sender,Receiver,Amount,source
0,9NWaxvJMV1z2U7d2B3nrC9pejhbSdfVfiTowEChSpump,5kufsc6NQ67LBpbTbvdecAMB5uMz6Mmb8vxtiRTpW8isUY...,C6RTwZMZ9tNYk69chMtUTSBvofWQAZD6AseZ92eg3oPY,8v2y9MRQyvVH6kQRAMyjiiKVyWbcAjpjTDATW2Y3yXFd,1.912000e+03,safe
1,9NWaxvJMV1z2U7d2B3nrC9pejhbSdfVfiTowEChSpump,4nqN8UMnmJkvFqDLvt8STTp7Za8qMcSLQ8Vis2EAHfJ2fu...,8v2y9MRQyvVH6kQRAMyjiiKVyWbcAjpjTDATW2Y3yXFd,C6RTwZMZ9tNYk69chMtUTSBvofWQAZD6AseZ92eg3oPY,1.812000e+03,safe
2,9NWaxvJMV1z2U7d2B3nrC9pejhbSdfVfiTowEChSpump,5udYmduByyPSPMUJdjEs5qRGY7UDh2rXb8sQbhxYgTBXpx...,C6RTwZMZ9tNYk69chMtUTSBvofWQAZD6AseZ92eg3oPY,Hb5ZzGnB7GtwEhGCV5vRwJ7YSzmPGGf68WMDxQ3QU3Sm,1.590000e+03,safe
3,9NWaxvJMV1z2U7d2B3nrC9pejhbSdfVfiTowEChSpump,3kiwkJLMVCwD2bmCx2RFf7BGkhY3a5mEF68CnvKjQpG81r...,Hb5ZzGnB7GtwEhGCV5vRwJ7YSzmPGGf68WMDxQ3QU3Sm,C6RTwZMZ9tNYk69chMtUTSBvofWQAZD6AseZ92eg3oPY,1.490000e+03,safe
4,9NWaxvJMV1z2U7d2B3nrC9pejhbSdfVfiTowEChSpump,qzg2jj1KPeT4BNbY3ARSNsQjwDGBub2EXtZZMDu4jhWEpm...,FT5rhdp9Uze9eeoamjvFiBnK3Fih8Dc1827K2ht5Hqqm,4Jao4LimfYHjDPauXgjrngYxoHWxhVVUMk2CqTZzvMA9,6.117173e+06,safe
...,...,...,...,...,...,...
7653,6tCac1mPp8wVNURCDJFwGQNgcBsDN7m1gDQR5BL7pump,247b5RQTG7Bn6U5kyqjhpNNR3HiwJMfyxWuRx19nkDPbtV...,CgEN1nrA6HwK5QP8w6tUw8JH29a2KGbqFXcGYTvPTEDZ,AqKxy3aSKfqGP2DUJEoK6e1sSc6xCtgYPJvJR5hh8rQD,1.336859e+06,safe
7654,Kp18NinNncDZoCajmHE2t4iNUGr2Y89ph98FkF7pump,4yqUx8awJkfHP6fDm54VJCVY1UVsZurFfRaYUKV7x3X9eD...,ECAZj39L4f79WSqQYgtQCfS9ubqzpWX5fUC1mumUd4wQ,G9dmCJzZ9da2oNv7AYbmUXQ4JARWWegQCHmuCAwhYKno,5.442596e+06,safe
7655,Kp18NinNncDZoCajmHE2t4iNUGr2Y89ph98FkF7pump,2S2o2qUFKWJXDDNCsWsMQJB7mMfuUUvWhQnfj734uoFtdR...,Euyx7qvDNNoSsDf4PRwondGvFyqufPLRifxNABZuphCw,G9dmCJzZ9da2oNv7AYbmUXQ4JARWWegQCHmuCAwhYKno,1.656420e+06,safe
7656,Kp18NinNncDZoCajmHE2t4iNUGr2Y89ph98FkF7pump,29MfpdsLUnSFVUJWpfqKmSSybPTbATT2xVdBLpx2aR6LT7...,71P2fVQKXBvJCn7369izX1vva9JHQGMPtzCmMMDeVKyC,G9dmCJzZ9da2oNv7AYbmUXQ4JARWWegQCHmuCAwhYKno,1.807448e+06,safe


In [6]:
print(f"All wallets in edges_df:   {len(all_wallets):,}")
print(f"Crawled in origins_df:     {len(crawled_wallets):,}")
print(f"Uncrawled total:           {len(uncrawled):,}")
print(f"  → in puppet_wallets:     {len(uncrawled_scam):,}")
print(f"  → NOT in puppet_wallets: {len(uncrawled_organic):,}")


All wallets in edges_df:   3,271
Crawled in origins_df:     3,268
Uncrawled total:           3
  → in puppet_wallets:     0
  → NOT in puppet_wallets: 3


In [3]:
# ── For wallets NOT in origins_df, generate unique organic-looking defaults ──
all_wallets_in_graph = set(edges_df['Sender'].tolist() + edges_df['Receiver'].tolist())
crawled_wallets      = set(origins_df['Wallet'].tolist())
uncrawled_wallets    = all_wallets_in_graph - crawled_wallets

# Each uncrawled wallet gets its OWN random organic values
uncrawled_rows = []
for w in uncrawled_wallets:
    uncrawled_rows.append({
        'Wallet':          w,
        'Genesis_Signature': 'unknown',
        'Funder_Wallet':   'Unknown',
        'blockTime':       int(np.random.uniform(BASE_EPOCH, BASE_EPOCH + 365*24*3600)),
        'Funding_Amount':  round(np.random.uniform(0.5, 20.0), 4)
    })

uncrawled_df = pd.DataFrame(uncrawled_rows)
origins_full = pd.concat([origins_df, uncrawled_df], ignore_index=True)
origins_full.to_csv("../exported_data/wallet_genesis_origins_enriched.csv", index=False)

print(f"Crawled: {len(origins_df)} | Uncrawled filled: {len(uncrawled_rows)}")


Crawled: 3268 | Uncrawled filled: 3
